In [1]:
import pandas as pd
import numpy as np
import warnings
import joblib
import json
from pathlib import Path
import sys, os
from datetime import datetime
from glob import glob

project_root = Path.cwd()
while not (project_root / "src").exists():
    project_root = project_root.parent
project_root_str = str(project_root)
if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)
os.chdir(project_root_str)

from src.utils.team_info import nameDict
from src.utils.arbitrage import * 
from src.utils.ev_betting import *

warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)
np.random.seed(42)

today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

print("Setup complete.")
print(f"Date: {current_date}")

Setup complete.
Date: 2026-04-18


In [2]:
def get_latest_file(pattern):
    files = list(Path('data/raw/player_lines').glob(pattern))
    if not files:
        return None
    return max(files, key=lambda p: p.stat().st_mtime)

nba_us_file = get_latest_file(f'NBA_US_{today}*.csv')

In [3]:
def get_latest_file(pattern):
    files = list(Path('data/raw/mlb_player_lines').glob(pattern))
    if not files:
        return None
    return max(files, key=lambda p: p.stat().st_mtime)

mlb_us_file = get_latest_file(f'MLB_US_{today}*.csv')

In [4]:
def get_latest_file(pattern):
    files = list(Path('data/raw/afl_player_lines').glob(pattern))
    if not files:
        return None
    return max(files, key=lambda p: p.stat().st_mtime)

afl_us_file = get_latest_file(f'AFL_AU_{today}*.csv')

In [5]:
def get_latest_file(pattern):
    files = list(Path('data/raw/soccer_player_lines').glob(pattern))
    if not files:
        return None
    return max(files, key=lambda p: p.stat().st_mtime)

soccer_us_file = get_latest_file(f'SOCCER_US_{today}*.csv')

In [6]:
def get_latest_file(pattern):
    files = list(Path('data/raw/team_lines').glob(pattern))
    return max(files, key=lambda f: f.stat().st_mtime) if files else None

file = get_latest_file('NBA_*.json')
if file is None:
    raise ValueError("No JSON file found")

# try normal load first
try:
    team_dds = pd.read_json(file)
except ValueError:
    # fallback for nested JSON
    import json
    with open(file) as f:
        data = json.load(f)
    team_dds = pd.json_normalize(data)

print("Loaded:", file.name)
team_dds.head()

Loaded: NBA_20260418_185230.json


,home_team,away_team,commence_time,bookmakers
0,Los Angeles Lakers,Houston Rockets,2026-04-19 00:48:12+00:00,"[{'bookmaker': 'DraftKings', 'last_updated': '..."
1,Boston Celtics,Philadelphia 76ers,2026-04-19 17:10:00+00:00,"[{'bookmaker': 'DraftKings', 'last_updated': '..."
2,Oklahoma City Thunder,Phoenix Suns,2026-04-19 19:40:00+00:00,"[{'bookmaker': 'DraftKings', 'last_updated': '..."
3,Detroit Pistons,Orlando Magic,2026-04-19 22:40:00+00:00,"[{'bookmaker': 'DraftKings', 'last_updated': '..."
4,San Antonio Spurs,Portland Trail Blazers,2026-04-20 01:10:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."


In [7]:
nba_prop_arbs = scan_player_props(nba_us_file) if nba_us_file is not None else []
team_arbs = scan_team_odds(team_dds)

nba_all_arbs = pd.DataFrame(nba_prop_arbs + team_arbs).sort_values("margin_%", ascending=False) \
    if (nba_prop_arbs or team_arbs) else pd.DataFrame()

print(f"Player prop arbs: {len(nba_prop_arbs)} | Team odds arbs: {len(team_arbs)}")
nba_all_arbs

Player prop arbs: 55 | Team odds arbs: 0


,type,market,side_a,side_b,margin_%,stake_a,stake_b,profit
37,prop,Luke Kennard player_assists 4.5,Over @ theScore Bet (+600),Under @ Pinnacle (-111),33.108,2.14,7.86,4.95
44,prop,Reed Sheppard player_assists 4.5,Over @ Pinnacle (+124),Under @ FanDuel (+128),11.497,5.04,4.96,1.30
38,prop,Luke Kennard player_rebounds 3.5,Over @ Pinnacle (+108),Under @ Bovada (+145),11.107,5.41,4.59,1.25
3,prop,Alperen Sengun player_rebounds 8.5,Over @ FanDuel (+122),Under @ Pinnacle (+110),7.336,4.86,5.14,0.79
48,prop,Rui Hachimura player_rebounds 3.5,Over @ Pinnacle (-114),Under @ Bovada (+145),5.913,5.66,4.34,0.63
51,prop,Tari Eason player_rebounds 5.5,Over @ Pinnacle (-112),Under @ DraftKings (+140),5.503,5.59,4.41,0.58
45,prop,Reed Sheppard player_rebounds 2.5,Over @ Hard Rock Bet (+115),Under @ Pinnacle (+105),4.708,4.88,5.12,0.49
34,prop,Josh Okogie player_rebounds 1.5,Over @ Fliff (+105),Under @ Hard Rock Bet (FL) (+115),4.708,5.12,4.88,0.49
24,prop,Jabari Smith Jr player_rebounds 7.5,Over @ Bovada (+125),Under @ DraftKings (-105),4.336,4.65,5.35,0.45
32,prop,Jordan Goodwin player_points 8.5,Over @ Hard Rock Bet (FL) (+105),Under @ betPARX (+112),4.050,5.08,4.92,0.42


In [8]:
mlb_prop_arbs = scan_player_props(mlb_us_file) if mlb_us_file is not None else []

mlb_all_arbs = pd.DataFrame(mlb_prop_arbs + team_arbs).sort_values("margin_%", ascending=False) \
    if (mlb_prop_arbs or team_arbs) else pd.DataFrame()

print(f"Player prop arbs: {len(mlb_prop_arbs)}")
mlb_all_arbs

Player prop arbs: 17


,type,market,side_a,side_b,margin_%,stake_a,stake_b,profit
2,prop,Emmet Sheehan pitcher_hits_allowed 4.5,Over @ BetOnline.ag (+110),Under @ BetMGM (+115),5.869,5.06,4.94,0.62
15,prop,Shohei Ohtani batter_stolen_bases 0.5,Over @ Bovada (+475),Under @ DraftKings (-372),3.795,1.81,8.19,0.39
8,prop,JJ Wetherholt batter_hits 0.5,Over @ BetMGM (-135),Under @ BetOnline.ag (+148),2.231,5.88,4.12,0.23
0,prop,Chase DeLauter batter_runs_scored 0.5,Over @ BetMGM (+275),Under @ DraftKings (-248),2.069,2.72,7.28,0.21
1,prop,Eloy Jimenez batter_singles 0.5,Over @ DraftKings (+112),Under @ BetMGM (-105),1.611,4.79,5.21,0.16
10,prop,Myles Straw batter_hits 0.5,Over @ BetMGM (-120),Under @ DraftKings (+128),1.595,5.54,4.46,0.16
9,prop,Mickey Moniak batter_hits 0.5,Over @ BetOnline.ag (-156),Under @ BetMGM (+165),1.327,6.18,3.82,0.13
7,prop,Ivan Herrera batter_hits 0.5,Over @ BetMGM (-160),Under @ BetOnline.ag (+169),1.287,6.23,3.77,0.13
6,prop,Freddie Freeman batter_total_bases 1.5,Over @ BetMGM (-120),Under @ DraftKings (+126),1.207,5.52,4.48,0.12
11,prop,Ozzie Albies batter_hits 1.5,Over @ BetRivers (+265),Under @ BetMGM (-250),1.174,2.77,7.23,0.12


In [9]:
afl_prop_arbs = scan_player_props(afl_us_file) if afl_us_file is not None else []

afl_all_arbs = pd.DataFrame(afl_prop_arbs).sort_values("margin_%", ascending=False) \
    if (afl_prop_arbs or team_arbs) else pd.DataFrame()

print(f"Player prop arbs: {len(afl_prop_arbs)}")
afl_all_arbs

Player prop arbs: 0


""


In [10]:
soccer_prop_arbs = scan_player_props(soccer_us_file) if soccer_us_file is not None else []

soccer_all_arbs = pd.DataFrame(soccer_prop_arbs).sort_values("margin_%", ascending=False) \
    if (soccer_prop_arbs or team_arbs) else pd.DataFrame()

print(f"Player prop arbs: {len(soccer_prop_arbs)}")
soccer_all_arbs

Player prop arbs: 9


,type,market,side_a,side_b,margin_%,stake_a,stake_b,profit
8,prop,Yeremi Pino player_shots_on_target 0.5,Over @ BetRivers (+165),Under @ BetOnline.ag (+110),14.645,4.42,5.58,1.72
1,prop,Daniel Munoz player_shots_on_target 0.5,Over @ BetRivers (+260),Under @ BetOnline.ag (-179),8.065,3.02,6.98,0.88
2,prop,Jean-Philippe Mateta player_shots_on_target 1.5,Over @ BetRivers (+210),Under @ BetOnline.ag (-175),4.106,3.36,6.64,0.43
6,prop,Morgan Rogers player_shots_on_target 0.5,Over @ BetRivers (-159),Under @ BetOnline.ag (+175),2.246,6.28,3.72,0.23
7,prop,Ollie Watkins player_shots_on_target 1.5,Over @ BetRivers (+220),Under @ BetOnline.ag (-200),2.083,3.19,6.81,0.21
4,prop,John McGinn player_shots_on_target 0.5,Over @ BetRivers (+190),Under @ BetOnline.ag (-175),1.881,3.51,6.49,0.19
0,prop,Bernardo Silva player_shots 1.5,Over @ BetRivers (+300),Under @ BetOnline.ag (-278),1.455,2.54,7.46,0.15
5,prop,Matty Cash player_shots 1.5,Over @ BetRivers (+225),Under @ BetOnline.ag (-217),0.777,3.10,6.90,0.08
3,prop,Jeremy Doku player_shots_on_target 0.5,Over @ BetRivers (+155),Under @ BetOnline.ag (-152),0.467,3.94,6.06,0.05


In [11]:

ev_bets = scan_plus_ev(
    team_file=None,
    props_file=nba_us_file,
    bankroll=1000,
    min_ev=2.0,      # only show bets with >2% EV
)
print_ev_results(ev_bets, bankroll=100)


  +EV SCANNER  |  bankroll=$100
  +EV bets found: 61

[PROP] Luke Kennard — player_assists Over 4.5
  Book: theScore Bet  |  Odds: +600
  True prob: 49.8%  vs  Implied: 14.3%
  EV: +248.48%  |  Kelly stake: $103.54

[PROP] Luke Kennard — player_rebounds Under 3.5
  Book: Bovada  |  Odds: +145
  True prob: 54.1%  vs  Implied: 40.8%
  EV: +32.59%  |  Kelly stake: $56.19

[PROP] Reed Sheppard — player_assists Under 4.5
  Book: FanDuel  |  Odds: +128
  True prob: 57.3%  vs  Implied: 43.9%
  EV: +30.73%  |  Kelly stake: $60.02

[PROP] Alperen Sengun — player_rebounds Over 8.5
  Book: FanDuel  |  Odds: +122
  True prob: 54.5%  vs  Implied: 45.0%
  EV: +21.03%  |  Kelly stake: $43.1

[PROP] Rui Hachimura — player_rebounds Under 3.5
  Book: Bovada  |  Odds: +145
  True prob: 49.1%  vs  Implied: 40.8%
  EV: +20.38%  |  Kelly stake: $35.13

[PROP] Tari Eason — player_rebounds Under 5.5
  Book: DraftKings  |  Odds: +140
  True prob: 49.6%  vs  Implied: 41.7%
  EV: +18.96%  |  Kelly stake: $33.86